# Pipeline de datos: `Dataset` y `DataLoader` en PyTorch

**Objetivos**

1. Leer los conjuntos experimentales sin volver a dividir las imágenes.
2. Preparar imágenes y máscaras con **OpenCV + Albumentations**, conservando la relación de aspecto.
3. Aplicar *data augmentation* **solo a entrenamiento**.
4. Implementar `Dataset` (`__init__`, `__len__`, `__getitem__`) y crear `DataLoader`.
5. Inspeccionar dimensiones, tipos, etiquetas y correspondencia visual imagen–máscara.

> **Requisitos:** `numpy`, `pandas`, `opencv-python`, `albumentations` (2.x), `torch`, `matplotlib` y Jupyter. Ejecutar el notebook desde la carpeta `notebooks/` del proyecto.

## 1. Importar bibliotecas

In [ ]:
# Importar bibliotecas necesarias
from pathlib import Path
import json
import numpy as np
import pandas as pd
import cv2
import albumentations as A
import torch
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from torch.utils.data import Dataset, DataLoader

## 2. Leer la configuración del proyecto

In [ ]:
# Leer el archivo de configuración
PROYECTO = Path("..").resolve()
ruta_config = PROYECTO / "config" / "dataset.json"
with ruta_config.open("r", encoding="utf-8") as archivo:
    config = json.load(archivo)

# Definir rutas de archivos y directorios
RUTA_MANIFEST = PROYECTO / config["manifest"]["path"]
DIRECTORIO_DATA = PROYECTO / "data"
DIRECTORIO_DATASET = PROYECTO / config["dataset"]["root"]
DIRECTORIO_IMAGENES = DIRECTORIO_DATASET / config["dataset"]["images"]
DIRECTORIO_SPLITS = PROYECTO / config["splits"]["path"]

print("Directorio de imágenes:", DIRECTORIO_IMAGENES)
print("Directorio de particiones:", DIRECTORIO_SPLITS)

## 3. Cargar las particiones experimentales

Usaremos el CSV **70/15/15** generado en una práctica anterior. No ejecutamos nuevamente `train_test_split`: las particiones ya están fijadas y son reproducibles.

In [ ]:
# Seleccionar una configuración experimental
splits = pd.read_csv(DIRECTORIO_SPLITS / "split_70_15_15.csv")

# Separar los registros
train_df = splits[splits["split"] == "train"].reset_index(drop=True)
val_df = splits[splits["split"] == "validation"].reset_index(drop=True)
test_df = splits[splits["split"] == "test"].reset_index(drop=True)

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

display(splits.head())

## 4. Definir los pipelines de Albumentations

**Procesamiento común:**

- `LongestMaxSize`: escala la imagen hasta que el lado más largo mida como máximo 256 px, **sin deformarla**.
- `PadIfNeeded`: completa hasta 256 × 256 con negro (`fill=0`) en la imagen.
- En las máscaras originales de Oxford-IIIT Pet, **el fondo tiene etiqueta 2**. Por eso el padding de la máscara debe ser `fill_mask=2`, no cero.

**Solo entrenamiento:** cuatro transformaciones independientes con `p=0.5` cada una. Las transformaciones geométricas se aplican de forma sincronizada a imagen y máscara; brillo y contraste solo modifican la imagen.

> Una probabilidad de 50 % **no** significa que se aplique exactamente la mitad de las veces en un conjunto pequeño; cada llamada realiza una decisión aleatoria independiente.

In [ ]:
TAMANO = 256

# Transformaciones comunes a train, validation y test (resize y padding)
transform_base = [
    A.LongestMaxSize(
        max_size=TAMANO,
        interpolation=cv2.INTER_LINEAR,
        mask_interpolation=cv2.INTER_NEAREST
    ),
    A.PadIfNeeded(
        min_height=TAMANO,
        min_width=TAMANO,
        border_mode=cv2.BORDER_CONSTANT,
        fill=0,        # Negro en la imagen RGB
        fill_mask=2,   # Fondo en el trimap original (1=animal, 2=fondo, 3=contorno)
        position="center"
    )
]

# Entrenamiento: procesamiento común + aumentos aleatorios
transform_train = A.Compose(transform_base + [
    A.Rotate(
        limit=20,
        interpolation=cv2.INTER_LINEAR,
        mask_interpolation=cv2.INTER_NEAREST,
        border_mode=cv2.BORDER_CONSTANT,
        fill=0,
        fill_mask=2,
        p=0.5
    ),
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(
        brightness_limit=0.20,
        contrast_limit=0,
        p=0.5
    ),
    A.RandomBrightnessContrast(
        brightness_limit=0,
        contrast_limit=0.20,
        p=0.5
    )
])

# Validación y prueba: solo procesamiento determinista (resize y padding)
transform_eval = A.Compose(transform_base)

### Nota sobre el orden de las transformaciones

Primero se ajusta el tamaño y se añade padding; después se aplican los aumentos. Al rotar una imagen ya cuadrada, las esquinas nuevas también se rellenan de negro en RGB y con la clase fondo en la máscara.

## 5. Implementar la clase `Dataset`

- `__init__`: guarda el DataFrame, la carpeta base y el pipeline de transformaciones.
- `__len__`: indica cuántas muestras contiene el conjunto.
- `__getitem__`: localiza **una** muestra, lee imagen y máscara, aplica transformaciones sincronizadas y devuelve tensores.

**Formatos:** imagen `float32` en `[0, 1]` con forma `(C, H, W)`; máscara `int64` con forma `(H, W)` y clases `0, 1, 2`.

> La conversión de etiquetas `1,2,3 → 0,1,2` supone que se usan los trimaps originales del minidataset, no máscaras binarias.

In [ ]:
class PetSegmentationDataset(Dataset):

    def __init__(self, dataframe, directorio_imagenes, transform):
        self.dataframe = dataframe.reset_index(drop=True)
        self.directorio_imagenes = Path(directorio_imagenes)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        # 1. Seleccionar una fila del CSV
        fila = self.dataframe.iloc[idx]

        # 2. Construir rutas (el CSV contiene rutas relativas a images/)
        ruta_imagen = self.directorio_imagenes / fila["image_path"]
        ruta_mascara = self.directorio_imagenes / fila["mask_path"]

        # 3. Leer imagen BGR y máscara de etiquetas
        imagen = cv2.imread(str(ruta_imagen), cv2.IMREAD_COLOR)
        mascara = cv2.imread(str(ruta_mascara), cv2.IMREAD_GRAYSCALE)

        if imagen is None:
            raise FileNotFoundError(f"No se pudo leer la imagen: {ruta_imagen}")
        if mascara is None:
            raise FileNotFoundError(f"No se pudo leer la máscara: {ruta_mascara}")

        # 4. Convertir BGR a RGB
        imagen = cv2.cvtColor(imagen, cv2.COLOR_BGR2RGB)

        # 5. Aplicar transformaciones sincronizadas
        resultado = self.transform(image=imagen, mask=mascara)
        imagen = resultado["image"]
        mascara = resultado["mask"]

        # 6. Convertir tipos y normalizar imagen
        imagen = imagen.astype(np.float32) / 255.0
        mascara = mascara.astype(np.int64) - 1

        # 7. Convertir HWC -> CHW y construir tensores
        imagen = torch.from_numpy(imagen.transpose(2, 0, 1).copy())
        mascara = torch.from_numpy(mascara.copy())

        return imagen, mascara

## 6. Crear los tres objetos `Dataset`

In [ ]:
train_dataset = PetSegmentationDataset(
    train_df, DIRECTORIO_IMAGENES, transform=transform_train
)

val_dataset = PetSegmentationDataset(
    val_df, DIRECTORIO_IMAGENES, transform=transform_eval
)

test_dataset = PetSegmentationDataset(
    test_df, DIRECTORIO_IMAGENES, transform=transform_eval
)

print("Muestras de entrenamiento:", len(train_dataset))
print("Muestras de validación:", len(val_dataset))
print("Muestras de prueba:", len(test_dataset))

## 7. Recuperar una muestra y comprobar los tensores

`dataset[0]` llama internamente a `__getitem__(0)`. En entrenamiento, solicitar dos veces la misma muestra puede producir resultados visuales distintos debido al augmentation.

In [ ]:
imagen, mascara = train_dataset[0]

print("Imagen:", imagen.shape, imagen.dtype)
print("Máscara:", mascara.shape, mascara.dtype)
print("Etiquetas presentes:", torch.unique(mascara).tolist())

assert imagen.dtype == torch.float32
assert mascara.dtype == torch.int64
assert 0 <= imagen.min().item() <= imagen.max().item() <= 1
assert set(torch.unique(mascara).tolist()).issubset({0, 1, 2})
print("Verificación correcta")

## 8. Visualizar imagen y máscara

La imagen se devuelve en formato **CHW** para PyTorch; `imshow` necesita **HWC**, por eso utilizamos `permute(1, 2, 0)`. La máscara no se normaliza ni se interpola con bilineal.

In [ ]:
cmap = ListedColormap(["#D0805C", "#F7F5F1", "#568A80"])

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
axes[0].imshow(imagen.permute(1, 2, 0))
axes[0].set_title("Imagen RGB")
axes[1].imshow(mascara, cmap=cmap, vmin=0, vmax=2, interpolation="nearest")
axes[1].set_title("Máscara (0=animal, 1=fondo, 2=contorno)")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

## 9. Comparar augmentation y procesamiento determinista

Las dos filas superiores muestran **dos llamadas** a la misma imagen de entrenamiento: pueden cambiar por la rotación, flip, brillo o contraste. La fila inferior corresponde a validación, sin aumentos aleatorios.

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(9, 12))

for i, (nombre, dataset) in enumerate([
    ("Train: llamada 1", train_dataset),
    ("Train: llamada 2", train_dataset),
    ("Validation", val_dataset)
]):
    img, mask = dataset[0]
    axes[i, 0].imshow(img.permute(1, 2, 0))
    axes[i, 0].set_title(f"{nombre} — imagen")
    axes[i, 1].imshow(mask, cmap=cmap, vmin=0, vmax=2, interpolation="nearest")
    axes[i, 1].set_title(f"{nombre} — máscara")
    for ax in axes[i]:
        ax.axis("off")

plt.tight_layout()
plt.show()

## 10. Crear los `DataLoader`

`Dataset` sabe recuperar **una muestra**; `DataLoader` agrupa muestras en **batches** y puede mezclarlas.

- `batch_size=4`: hasta cuatro muestras por batch.
- `shuffle=True` **solo en entrenamiento**.
- `num_workers=0`: recomendado para esta introducción en notebooks y Windows.
- El último batch puede contener menos de cuatro muestras (`drop_last=False`, valor predeterminado).

In [ ]:
BATCH_SIZE = 4

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0
)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0
)

for nombre, dataset, loader in [
    ("Train", train_dataset, train_loader),
    ("Validation", val_dataset, val_loader),
    ("Test", test_dataset, test_loader)
]:
    print(f"{nombre}: {len(dataset)} muestras, {len(loader)} batches")

## 11. Inspeccionar un batch

`next(iter(train_loader))` obtiene el primer batch. Sus dimensiones son **BCHW** para imágenes y **BHW** para máscaras.

In [ ]:
imagenes, mascaras = next(iter(train_loader)) # Obtener un batch de datos del DataLoader de entrenamiento

print("Batch de imágenes:", imagenes.shape, imagenes.dtype)
print("Batch de máscaras:", mascaras.shape, mascaras.dtype)
print("Etiquetas del batch:", torch.unique(mascaras).tolist())

assert imagenes.ndim == 4
assert mascaras.ndim == 3
assert imagenes.shape[1:] == (3, TAMANO, TAMANO)
assert mascaras.shape[1:] == (TAMANO, TAMANO)
assert imagenes.shape[0] == mascaras.shape[0]
assert imagenes.dtype == torch.float32
assert mascaras.dtype == torch.int64
assert set(torch.unique(mascaras).tolist()).issubset({0, 1, 2})
print("Batch verificado correctamente")

## 12. Visualizar las muestras de un batch

In [ ]:
n = imagenes.shape[0]
fig, axes = plt.subplots(n, 2, figsize=(8, 3.5 * n), squeeze=False)

for i in range(n):
    axes[i, 0].imshow(imagenes[i].permute(1, 2, 0))
    axes[i, 0].set_title(f"Imagen {i + 1}")
    axes[i, 1].imshow(mascaras[i], cmap=cmap, vmin=0, vmax=2, interpolation="nearest")
    axes[i, 1].set_title(f"Máscara {i + 1}")
    for ax in axes[i]:
        ax.axis("off")

plt.tight_layout()
plt.show()

## 13. Punto de control

1. Cambia el CSV a `split_80_10_10.csv` y ejecuta de nuevo el notebook desde la celda de carga de particiones.
2. Compara el número de muestras y batches entre las dos configuraciones.
3. Ejecuta dos veces la celda de visualización de entrenamiento: ¿se obtienen las mismas imágenes transformadas? ¿Por qué?
4. Comprueba que las máscaras siguen teniendo **solo etiquetas enteras** `0`, `1`, `2`.
5. Explica por qué **no** se aplican rotaciones, flips, brillo ni contraste aleatorios a validación y prueba.

**Resultado esperado:** tres datasets y tres dataloaders con imágenes y máscaras alineadas, dimensiones correctas y augmentation exclusivo para entrenamiento.